# Parameter Optimization

Grid-search key strategy parameters and visualise how each affects performance.

**Sections**
1. Setup
2. Confidence threshold sweep
3. Position size sweep
4. 2-D grid search (threshold × position size)
5. Risk profile comparison
6. Best parameters summary

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.automated_trading_system import AutomatedTradingSystem

TICKER     = 'AAPL'
START_DATE = '2023-01-01'
END_DATE   = '2024-01-01'
CAPITAL    = 10_000

os.makedirs('../runs', exist_ok=True)


def run(ticker=TICKER, capital=CAPITAL, confidence_threshold=0.55,
        max_position_size_pct=0.05, max_positions=5, max_daily_loss_pct=0.02):
    """Run a single backtest and return the result dict."""
    s = AutomatedTradingSystem(
        initial_capital=capital,
        ticker=ticker,
        max_positions=max_positions,
        max_position_size_pct=max_position_size_pct,
    )
    s.signal_generator.confidence_threshold = confidence_threshold
    s.risk_manager.max_daily_loss_pct = max_daily_loss_pct
    r = s.backtest(start_date=START_DATE, end_date=END_DATE)
    return {
        'return_pct'    : r['portfolio']['return_pct'],
        'win_rate'      : r['trades']['win_rate'],
        'profit_factor' : r['trades'].get('profit_factor', 0),
        'total_trades'  : r['trades']['total_trades'],
        'signals'       : r['signals']['total_signals'],
    }


print('Ready. Each run is a full backtest — larger grids take longer.')

## 2. Confidence Threshold Sweep

Higher threshold → fewer but higher-quality signals.

In [ ]:
thresholds = [0.50, 0.52, 0.55, 0.58, 0.60, 0.63, 0.65, 0.70]
thresh_rows = []

for thr in thresholds:
    print(f'  threshold={thr:.2f}...', end=' ')
    r = run(confidence_threshold=thr)
    r['threshold'] = thr
    thresh_rows.append(r)
    print(f'return={r["return_pct"]:+.1f}%  trades={r["total_trades"]}')

thresh_df = pd.DataFrame(thresh_rows).set_index('threshold')
print('\n', thresh_df.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f'{TICKER} — Confidence Threshold Sweep', fontsize=12, fontweight='bold')

pairs = [
    ('return_pct',    'Return (%)',     'steelblue'),
    ('win_rate',      'Win Rate (%)',   'green'),
    ('profit_factor', 'Profit Factor',  'darkorange'),
    ('total_trades',  'Total Trades',   'purple'),
]
for ax, (col, label, color) in zip(axes.flat, pairs):
    ax.plot(thresh_df.index, thresh_df[col], color=color, linewidth=2, marker='o', markersize=6)
    ax.set_xlabel('Confidence Threshold'); ax.set_ylabel(label)
    ax.set_title(label); ax.grid(alpha=0.3)
    best_idx = thresh_df[col].idxmax() if col != 'total_trades' else thresh_df[col].idxmin()
    ax.axvline(best_idx, color='red', linewidth=0.8, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('../runs/nb_threshold_sweep.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: runs/nb_threshold_sweep.png')

## 3. Position Size Sweep

Larger position size amplifies both gains and losses.

In [ ]:
pos_sizes = [0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.20]
size_rows = []

for ps in pos_sizes:
    print(f'  position_size={ps:.0%}...', end=' ')
    r = run(max_position_size_pct=ps)
    r['position_size_pct'] = ps * 100
    size_rows.append(r)
    print(f'return={r["return_pct"]:+.1f}%  win_rate={r["win_rate"]:.0f}%')

size_df = pd.DataFrame(size_rows).set_index('position_size_pct')
print('\n', size_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'{TICKER} — Position Size Sweep', fontsize=12, fontweight='bold')

for ax, (col, label, color) in zip(axes, [
    ('return_pct',    'Return (%)',    'steelblue'),
    ('win_rate',      'Win Rate (%)',  'green'),
    ('profit_factor', 'Profit Factor', 'darkorange'),
]):
    ax.plot(size_df.index, size_df[col], color=color, linewidth=2, marker='s', markersize=6)
    ax.set_xlabel('Max Position Size (%)'); ax.set_ylabel(label)
    ax.set_title(label); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../runs/nb_position_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. 2-D Grid Search  (Threshold × Position Size)

In [ ]:
THRESH_GRID = [0.50, 0.55, 0.60, 0.65]
SIZE_GRID   = [0.05, 0.08, 0.12]

grid_returns = np.zeros((len(THRESH_GRID), len(SIZE_GRID)))
grid_wr      = np.zeros_like(grid_returns)

for i, thr in enumerate(THRESH_GRID):
    for j, ps in enumerate(SIZE_GRID):
        print(f'  thr={thr:.2f}  size={ps:.0%}...', end=' ')
        r = run(confidence_threshold=thr, max_position_size_pct=ps)
        grid_returns[i, j] = r['return_pct']
        grid_wr[i, j]      = r['win_rate']
        print(f'return={r["return_pct"]:+.1f}%')

print('Grid complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'{TICKER} — 2-D Grid Search', fontsize=12, fontweight='bold')

xlabels = [f'{p:.0%}' for p in SIZE_GRID]
ylabels = [f'{t:.2f}' for t in THRESH_GRID]

for ax, data, title, cmap in [
    (axes[0], grid_returns, 'Return (%)',   'RdYlGn'),
    (axes[1], grid_wr,      'Win Rate (%)', 'RdYlGn'),
]:
    im = ax.imshow(data, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(SIZE_GRID)));   ax.set_xticklabels(xlabels)
    ax.set_yticks(range(len(THRESH_GRID))); ax.set_yticklabels(ylabels)
    ax.set_xlabel('Max Position Size'); ax.set_ylabel('Confidence Threshold')
    ax.set_title(title)
    for (r, c), val in np.ndenumerate(data):
        ax.text(c, r, f'{val:+.1f}', ha='center', va='center', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)

plt.tight_layout()
plt.savefig('../runs/nb_grid_search.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: runs/nb_grid_search.png')

## 5. Risk Profile Comparison

Compare the three built-in risk profiles from `config/risk_profiles.yml`.

In [ ]:
import yaml

with open('../config/risk_profiles.yml') as f:
    profiles = yaml.safe_load(f)

profile_rows = []
for name, cfg in profiles.items():
    print(f'  Profile: {name}...', end=' ')
    r = run(
        max_position_size_pct = cfg.get('max_position_size_pct', 0.05),
        max_positions         = cfg.get('max_positions', 5),
        max_daily_loss_pct    = cfg.get('max_daily_loss_pct', 0.02),
        confidence_threshold  = cfg.get('confidence_threshold', 0.55),
    )
    r['profile'] = name
    profile_rows.append(r)
    print(f'return={r["return_pct"]:+.1f}%  win_rate={r["win_rate"]:.0f}%')

prof_df = pd.DataFrame(profile_rows).set_index('profile')
print('\n', prof_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Risk Profile Comparison', fontsize=12, fontweight='bold')

profile_names = list(prof_df.index)
palette = ['#2196F3', '#FF9800', '#F44336']  # conservative, moderate, aggressive

for ax, (col, label) in zip(axes, [
    ('return_pct',    'Return (%)'),
    ('win_rate',      'Win Rate (%)'),
    ('profit_factor', 'Profit Factor'),
]):
    bars = ax.bar(profile_names, prof_df[col], color=palette[:len(profile_names)], alpha=0.85)
    ax.set_title(label); ax.set_ylabel(label); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, prof_df[col]):
        y = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, y + abs(y) * 0.02,
                f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../runs/nb_risk_profiles.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Best Parameters Summary

In [ ]:
# Best threshold (highest return)
best_thr = thresh_df['return_pct'].idxmax()
best_thr_ret = thresh_df.loc[best_thr, 'return_pct']

# Best position size (highest return)
best_ps = size_df['return_pct'].idxmax()
best_ps_ret = size_df.loc[best_ps, 'return_pct']

# Best grid cell
flat_idx = np.argmax(grid_returns)
best_i, best_j = np.unravel_index(flat_idx, grid_returns.shape)
best_grid_thr = THRESH_GRID[best_i]
best_grid_ps  = SIZE_GRID[best_j]
best_grid_ret = grid_returns[best_i, best_j]

print('=== Optimization Results ===')
print(f'\n1-D Threshold sweep')
print(f'   Best threshold : {best_thr:.2f}  →  return={best_thr_ret:+.2f}%')

print(f'\n1-D Position size sweep')
print(f'   Best size      : {best_ps:.0f}%  →  return={best_ps_ret:+.2f}%')

print(f'\n2-D Grid search')
print(f'   Best cell      : threshold={best_grid_thr:.2f}, size={best_grid_ps:.0%}  →  return={best_grid_ret:+.2f}%')

print(f'\nRecommended configuration:')
print(f'   confidence_threshold  = {best_grid_thr}')
print(f'   max_position_size_pct = {best_grid_ps}')
print()
print('Note: optimize on out-of-sample data to avoid overfitting.')